# CMS Healthcare Fraud & Anomaly Detection Brochure Generator

## Business Challenge

CMS manages large public healthcare programs such as Medicare and Medicaid.

Healthcare claims can include normal billing, mistakes, unusual patterns, or possible fraud.  
This notebook uses an LLM to create a clear brochure about:

- What CMS is
- How healthcare claims work
- Types of CMS programs
- Types of healthcare claims
- Common fraud, waste, abuse, and anomaly patterns
- How fraud and anomalies can be detected
- Useful data sources and machine learning features
- How this supports healthcare AI and RAG applications

## Step 1: Import libraries

Run this cell first.

`truststore` helps fix common SSL certificate problems in Windows or Conda environments.

In [1]:
# If needed, install these packages first:
# !pip install openai beautifulsoup4 requests python-dotenv truststore certifi

import os
import json
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# Fix common Windows / Conda SSL certificate issue
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore loaded successfully")
except Exception as e:
    print("truststore was not loaded:", e)

truststore loaded successfully


## Step 2: Set up OpenAI and CMS website

In [2]:
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OPENAI_API_KEY found")
else:
    print("OPENAI_API_KEY not found. Please add it to your .env file.")

client = OpenAI()

MODEL = "gpt-4o-mini"
CMS_URL = "https://www.cms.gov/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

print("Model:", MODEL)
print("CMS URL:", CMS_URL)

OPENAI_API_KEY found
Model: gpt-4o-mini
CMS URL: https://www.cms.gov/


## Step 3: Read a webpage

This function reads a webpage and returns clean text.

It removes scripts, styles, images, and other content that is not useful for the LLM.

In [3]:
def get_website_text(url):
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    title = soup.title.string if soup.title else "No title found"

    if soup.body:
        for tag in soup.body(["script", "style", "img", "input", "noscript"]):
            tag.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""

    return f"Title:\n{title}\n\nContent:\n{text}"

### Test the webpage reader

In [4]:
cms_text = get_website_text(CMS_URL)

print(cms_text[:1000])

Title:
Home - Centers for Medicare & Medicaid Services | CMS

Content:
Skip to main content
Centers for Medicare & Medicaid Services
CMS Newsroom
CMS Global Secondary Menu
About CMS
Newsroom
Data & Research
Search CMS.gov
Search
Search
Popular terms
Physician Fee Schedule
Local Coverage Determination
Medically Unlikely Edits
Telehealth
Covid-19
Agents and Brokers
CMS.gov main menu
Medicare
Back to main menu
section title h2
Enrollment & renewal
Back to
menu
section title h3
Original Medicare (Part A and B) Eligibility and Enrollment
Annual Medicare Participation Announcement
Providers & suppliers
Medicare Managed Care Eligibility and Enrollment
Part D Eligibility and Enrollment
Health plans
Coverage
Back to
menu
section title h3
Coverage Determination Process
Medicare Coverage Database
Approved facilities, trials, & registries
Telehealth
Medicare Summary Notice
Prescription drug coverage contracting
Coverage with evidence development
Investigational device exemption studies
Prescriptio

## Step 4: Get links from a webpage

This function collects all links from the CMS homepage.

In [5]:
def get_website_links(url):
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    links = []

    for link in soup.find_all("a"):
        href = link.get("href")
        if href:
            links.append(href)

    return links

### Test the link function

In [6]:
links = get_website_links(CMS_URL)

print("Number of links:", len(links))
print("First 20 links:")
links[:20]

Number of links: 395
First 20 links:


['#skipNavTarget',
 '/',
 '/about-cms/contact/newsroom',
 '/about-cms',
 '/about-cms/contact/newsroom',
 '/data-research',
 '/medicare/enrollment-renewal/original-part-a-b',
 '/medicare-participation',
 '/medicare/enrollment-renewal/providers-suppliers',
 '/medicare/enrollment-renewal/managed-care-eligibility-enrollment',
 '/medicare/enrollment-renewal/part-d-enrollment-eligibility',
 '/medicare/enrollment-renewal/health-plans',
 '/medicare/coverage/determination-process',
 'https://www.cms.gov/medicare-coverage-database/search.aspx',
 '/medicare/coverage/approved-facilities-trials-registries',
 '/medicare/coverage/telehealth',
 '/medicare/coverage/summary-notice',
 '/medicare/coverage/prescription-drug-coverage-contracting',
 '/medicare/coverage/evidence',
 '/medicare/coverage/investigational-device-exemption-ide-studies']

## Step 5: Ask the LLM to select useful CMS links

The CMS homepage has many links.

We only want links that help create a brochure about healthcare claims, fraud, anomaly detection, billing, compliance, and program integrity.

In [7]:
link_system_prompt = """
You are a healthcare compliance analyst.

You will receive a list of links from a CMS webpage.

Select links that are useful for creating a brochure about:
CMS healthcare claims fraud and anomaly detection.

Select links related to:
- CMS overview
- Medicare
- Medicaid
- CHIP
- claims
- billing
- medical coding
- provider enrollment
- payment policy
- fraud prevention
- program integrity
- compliance
- audit
- overpayment
- healthcare regulations

Ignore links related to:
- privacy
- contact
- careers
- press releases
- events
- social media
- email
- accessibility statements

Return valid JSON only.

Use this format:

{
    "links": [
        {
            "type": "Medicare",
            "url": "https://www.cms.gov/example"
        }
    ]
}
"""

In [8]:
def get_links_prompt(url):
    links = get_website_links(url)
    user_prompt = f"""
Here are links found on this CMS webpage:

{url}

Choose only the links that are useful for creating a brochure about:
CMS healthcare claims fraud and anomaly detection.

Links:
"""

    user_prompt += "\n".join(links)

    return user_prompt

### Preview the link-selection prompt

In [9]:
print(get_links_prompt(CMS_URL)[:3000])


Here are links found on this CMS webpage:

https://www.cms.gov/

Choose only the links that are useful for creating a brochure about:
CMS healthcare claims fraud and anomaly detection.

Links:
#skipNavTarget
/
/about-cms/contact/newsroom
/about-cms
/about-cms/contact/newsroom
/data-research
/medicare/enrollment-renewal/original-part-a-b
/medicare-participation
/medicare/enrollment-renewal/providers-suppliers
/medicare/enrollment-renewal/managed-care-eligibility-enrollment
/medicare/enrollment-renewal/part-d-enrollment-eligibility
/medicare/enrollment-renewal/health-plans
/medicare/coverage/determination-process
https://www.cms.gov/medicare-coverage-database/search.aspx
/medicare/coverage/approved-facilities-trials-registries
/medicare/coverage/telehealth
/medicare/coverage/summary-notice
/medicare/coverage/prescription-drug-coverage-contracting
/medicare/coverage/evidence
/medicare/coverage/investigational-device-exemption-ide-studies
/medicare/coverage/prescription-drug-coverage
/med

In [10]:
def select_useful_links(url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_prompt(url)}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    result = response.choices[0].message.content
    return json.loads(result)

### Select useful CMS links

In [11]:
useful_links = select_useful_links(CMS_URL)

print(json.dumps(useful_links, indent=2)[:5000])

{
  "links": [
    {
      "type": "CMS Overview",
      "url": "https://www.cms.gov/about-cms"
    },
    {
      "type": "Medicare",
      "url": "/medicare/enrollment-renewal/providers-suppliers"
    },
    {
      "type": "Medicare",
      "url": "/medicare/regulations-guidance/manuals"
    },
    {
      "type": "Medicare",
      "url": "/medicare/coding-billing/icd-10-codes"
    },
    {
      "type": "Medicare",
      "url": "/medicare/coding-billing/national-correct-coding-initiative-ncci-edits"
    },
    {
      "type": "Medicare",
      "url": "/medicare/payment/fee-for-service-providers/part-b-drugs/average-drug-sales-price"
    },
    {
      "type": "Medicare",
      "url": "/medicare/payment/fee-schedules"
    },
    {
      "type": "Medicare",
      "url": "/medicare/medicaid-coordination/center-program-integrity/reporting-fraud"
    },
    {
      "type": "Medicaid",
      "url": "https://www.medicaid.gov"
    },
    {
      "type": "Medicaid",
      "url": "/medicaid-

## Step 6: Convert relative links into full URLs

Some links may look like this:

`/medicare`

This function converts them into full URLs:

`https://www.cms.gov/medicare`

In [12]:
def make_full_url(base_url, link):
    if not link:
        return None

    if link.startswith("http"):
        return link

    if link.startswith("/"):
        parts = base_url.split("/")
        domain = parts[0] + "//" + parts[2]
        return domain + link

    return base_url.rstrip("/") + "/" + link

### Test URL conversion

In [13]:
print(make_full_url(CMS_URL, "/medicare"))
print(make_full_url(CMS_URL, "https://www.cms.gov/medicaid"))
print(make_full_url(CMS_URL, "about-cms"))

https://www.cms.gov/medicare
https://www.cms.gov/medicaid
https://www.cms.gov/about-cms


## Step 7: Collect CMS content

This function collects:

1. CMS homepage content
2. A few useful CMS pages selected by the LLM

This combined content becomes the source material for the brochure.

In [14]:
def get_cms_content(url, max_links=25):
    content = "CMS Homepage:\n\n"
    content += get_website_text(url)

    useful_links = select_useful_links(url)
    selected_links = useful_links.get("links", [])[:max_links]

    print("Selected links:", len(selected_links))

    for item in selected_links:
        link_type = item.get("type", "Unknown")
        link_url = make_full_url(url, item.get("url"))

        if not link_url:
            continue

        print("Reading:", link_type, "-", link_url)

        try:
            content += "\n\n" + "=" * 80 + "\n"
            content += f"Page type: {link_type}\n"
            content += f"URL: {link_url}\n\n"
            content += get_website_text(link_url)
        except Exception as e:
            content += f"\n\nCould not read {link_url}: {e}"

    return content

### Preview collected CMS content

In [15]:
cms_content = get_cms_content(CMS_URL, max_links=3)

print("Content length:", len(cms_content))
print(cms_content[:4000])

Selected links: 3
Reading: CMS Overview - https://www.cms.gov/about-cms
Reading: Medicare - https://www.cms.gov/medicare/enrollment-renewal/providers-suppliers
Reading: Medicare - https://www.cms.gov/medicare/regulations-guidance/manuals
Content length: 69520
CMS Homepage:

Title:
Home - Centers for Medicare & Medicaid Services | CMS

Content:
Skip to main content
Centers for Medicare & Medicaid Services
CMS Newsroom
CMS Global Secondary Menu
About CMS
Newsroom
Data & Research
Search CMS.gov
Search
Search
Popular terms
Physician Fee Schedule
Local Coverage Determination
Medically Unlikely Edits
Telehealth
Covid-19
Agents and Brokers
CMS.gov main menu
Medicare
Back to main menu
section title h2
Enrollment & renewal
Back to
menu
section title h3
Original Medicare (Part A and B) Eligibility and Enrollment
Annual Medicare Participation Announcement
Providers & suppliers
Medicare Managed Care Eligibility and Enrollment
Part D Eligibility and Enrollment
Health plans
Coverage
Back to
menu
sec

## Step 8: Create the brochure prompt

This prompt tells the LLM exactly what the brochure should include.

In [16]:
brochure_system_prompt = """
You are a healthcare data scientist and healthcare compliance analyst.

Create a clear brochure about:

CMS Healthcare Claims Fraud & Anomaly Detection

Write in Markdown.

Use simple language that can be understood by:
- healthcare data scientists
- fraud analysts
- compliance teams
- AI engineers
- non-technical healthcare stakeholders

The brochure must include these sections:

# CMS Healthcare Claims Fraud & Anomaly Detection Brochure

## 1. Brief Introduction to CMS
Explain what CMS is.
Include Medicare, Medicaid, CHIP, and CMS's role in healthcare payment, quality, compliance, and regulation.

## 2. How Healthcare Claims Work
Explain the claim process:
patient visit, provider documentation, ICD diagnosis codes, CPT/HCPCS procedure codes,
claim submission, adjudication, payment, denial, and appeal.

Mention:
- CMS-1500 for professional claims
- UB-04 / CMS-1450 for institutional claims

## 3. Types of CMS Programs
Include:
- Medicare Part A
- Medicare Part B
- Medicare Advantage / Part C
- Medicare Part D
- Medicaid
- CHIP
- Dual eligibility
- Value-based care programs

## 4. Types of Healthcare Claims
Include:
- Professional claims
- Institutional claims
- Inpatient claims
- Outpatient claims
- Pharmacy claims
- Durable Medical Equipment claims
- Home health claims
- Hospice claims
- Laboratory claims
- Imaging claims
- Telehealth claims

## 5. Highlighted Fraud, Waste, Abuse, and Anomaly Types
Explain:
- Phantom billing
- Billing for services not rendered
- Upcoding
- Unbundling
- Duplicate billing
- Medical necessity fraud
- Identity theft
- Stolen beneficiary ID
- Kickbacks
- Provider collusion
- Prescription fraud
- DRG manipulation
- Modifier abuse
- Place-of-service abuse
- Excessive billing
- Overutilization
- Underutilization
- Abnormal provider behavior
- Suspicious patient behavior
- Abnormal billing patterns
- Outlier claims
- Impossible day billing
- Unusual diagnosis-procedure combinations
- Geographic anomalies
- Provider specialty mismatch

## 6. How to Detect Fraud or Anomalies
Explain:
- Rule-based detection
- Statistical analysis
- Peer-group comparison
- Outlier detection
- Isolation Forest
- Local Outlier Factor
- Logistic Regression
- Random Forest
- XGBoost
- Neural networks
- Time-series analysis
- Graph analytics
- Provider network analysis
- NLP
- Large Language Models
- RAG
- Knowledge graphs
- Explainable AI
- Human-in-the-loop review

## 7. Useful Machine Learning Features
Include:
- Provider ID / NPI
- Patient ID
- Diagnosis codes
- Procedure codes
- HCPCS/CPT
- DRG
- Revenue code
- NDC
- Claim amount
- Paid amount
- Service date
- Provider specialty
- Geography
- Billing frequency
- Historical utilization
- Referral patterns
- Denial history
- Medical necessity indicators

## 8. Useful CMS and Healthcare Data Sources
Include:
- CMS claims data
- Medicare data
- Medicaid data
- NPI Registry
- PECOS
- ICD-10
- CPT
- HCPCS
- DRG
- Revenue codes
- NDC
- Provider enrollment data
- OIG exclusion list
- Beneficiary data
- Payment policy documents

## 9. AI and Business Applications
Explain how this supports:
- Fraud detection
- Claims anomaly detection
- Payment integrity
- Claims auditing
- Provider risk scoring
- Compliance monitoring
- Investigation support
- Healthcare analytics
- RAG chatbot
- LLM healthcare assistant
- Human investigator workflow

Use headings, bullet points, and tables when helpful.
Make the brochure practical and portfolio-ready.
"""

## Step 9: Build the user prompt

This function combines the project instruction and CMS content.

In [17]:
def get_brochure_prompt(project_name, url, max_links=5):
    cms_content = get_cms_content(url, max_links=max_links)

    user_prompt = f"""
Project name:
{project_name}

Create a professional brochure using the CMS content below.

CMS content:
{cms_content}
"""

    return user_prompt[:120000]

### Preview the brochure prompt

In [18]:
preview_prompt = get_brochure_prompt(
    project_name="CMS Healthcare Claims Fraud & Anomaly Detection",
    url=CMS_URL,
    max_links=3
)

print("Prompt length:", len(preview_prompt))
print(preview_prompt[:4000])

Selected links: 3
Reading: CMS Overview - https://www.cms.gov/about-cms
Reading: Medicare - https://www.cms.gov/medicare/enrollment-renewal/providers-suppliers
Reading: Medicare - https://www.cms.gov/medicare/regulations-guidance/manuals
Prompt length: 69659

Project name:
CMS Healthcare Claims Fraud & Anomaly Detection

Create a professional brochure using the CMS content below.

CMS content:
CMS Homepage:

Title:
Home - Centers for Medicare & Medicaid Services | CMS

Content:
Skip to main content
Centers for Medicare & Medicaid Services
CMS Newsroom
CMS Global Secondary Menu
About CMS
Newsroom
Data & Research
Search CMS.gov
Search
Search
Popular terms
Physician Fee Schedule
Local Coverage Determination
Medically Unlikely Edits
Telehealth
Covid-19
Agents and Brokers
CMS.gov main menu
Medicare
Back to main menu
section title h2
Enrollment & renewal
Back to
menu
section title h3
Original Medicare (Part A and B) Eligibility and Enrollment
Annual Medicare Participation Announcement
Provid

## Step 10: Generate the brochure

In [19]:
def create_brochure(project_name, url, max_links=25):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_prompt(project_name, url, max_links=max_links)}
        ],
        temperature=0.2
    )

    brochure = response.choices[0].message.content

    display(Markdown(brochure))

    return brochure

### Create the final brochure

In [20]:
brochure = create_brochure(
    project_name="CMS Healthcare Claims Fraud & Anomaly Detection",
    url=CMS_URL,
    max_links=25
)

Selected links: 14
Reading: CMS Overview - https://www.cms.gov/about-cms
Reading: Medicare - https://www.cms.gov/medicare/enrollment-renewal/providers-suppliers
Reading: Medicare - https://www.cms.gov/medicare/regulations-guidance/manuals
Reading: Medicare - https://www.cms.gov/medicare/coding-billing/icd-10-codes
Reading: Medicare - https://www.cms.gov/medicare/coding-billing/national-correct-coding-initiative-ncci-edits
Reading: Medicare - https://www.cms.gov/medicare/payment/fee-for-service-providers/part-b-drugs/average-drug-sales-price
Reading: Medicare - https://www.cms.gov/medicare/payment/fee-schedules
Reading: Medicare - https://www.cms.gov/medicare/medicaid-coordination/center-program-integrity/reporting-fraud
Reading: Medicaid - https://www.medicaid.gov
Reading: Medicaid - https://www.cms.gov/medicaid-chip/medicare-coordination/integrity-institute
Reading: Fraud Prevention - https://www.cms.gov/fraud
Reading: Program Integrity - https://www.cms.gov/medicare/medicaid-coordina

```markdown
# CMS Healthcare Claims Fraud & Anomaly Detection Brochure

## 1. Brief Introduction to CMS
The **Centers for Medicare & Medicaid Services (CMS)** is a federal agency that provides health coverage to over 160 million Americans through programs like **Medicare**, **Medicaid**, and the **Children's Health Insurance Program (CHIP)**. CMS plays a crucial role in healthcare payment, quality, compliance, and regulation, ensuring that healthcare services are accessible and of high quality.

## 2. How Healthcare Claims Work
The healthcare claims process involves several steps:
- **Patient Visit**: A patient visits a healthcare provider for services.
- **Provider Documentation**: The provider documents the visit and services rendered.
- **ICD Diagnosis Codes**: The provider assigns ICD codes to the patient's diagnosis.
- **CPT/HCPCS Procedure Codes**: The provider assigns CPT or HCPCS codes for the services provided.
- **Claim Submission**: The provider submits the claim to CMS or an insurance company.
- **Adjudication**: The claim is reviewed for accuracy and compliance.
- **Payment**: If approved, payment is made to the provider.
- **Denial**: If denied, the provider may appeal the decision.

**Forms Used**:
- **CMS-1500**: For professional claims.
- **UB-04 / CMS-1450**: For institutional claims.

## 3. Types of CMS Programs
CMS administers various programs, including:
- **Medicare Part A**: Hospital insurance.
- **Medicare Part B**: Medical insurance.
- **Medicare Advantage / Part C**: Private insurance plans.
- **Medicare Part D**: Prescription drug coverage.
- **Medicaid**: State and federal program for low-income individuals.
- **CHIP**: Health coverage for children in families with incomes too high to qualify for Medicaid.
- **Dual Eligibility**: Individuals eligible for both Medicare and Medicaid.
- **Value-Based Care Programs**: Programs aimed at improving care quality and reducing costs.

## 4. Types of Healthcare Claims
Healthcare claims can be categorized as:
- **Professional Claims**: Services provided by individual practitioners.
- **Institutional Claims**: Services provided by hospitals or facilities.
- **Inpatient Claims**: For patients admitted to a hospital.
- **Outpatient Claims**: For services provided without an overnight stay.
- **Pharmacy Claims**: For prescription medications.
- **Durable Medical Equipment Claims**: For medical equipment.
- **Home Health Claims**: For home healthcare services.
- **Hospice Claims**: For end-of-life care.
- **Laboratory Claims**: For lab tests.
- **Imaging Claims**: For diagnostic imaging services.
- **Telehealth Claims**: For remote healthcare services.

## 5. Highlighted Fraud, Waste, Abuse, and Anomaly Types
Common types of fraud and anomalies include:
- **Phantom Billing**: Billing for services not provided.
- **Upcoding**: Charging for a more expensive service than was provided.
- **Unbundling**: Separating services that should be billed together.
- **Duplicate Billing**: Submitting the same claim multiple times.
- **Medical Necessity Fraud**: Billing for unnecessary services.
- **Identity Theft**: Using someone else's information to bill for services.
- **Kickbacks**: Receiving payment for referrals.
- **Provider Collusion**: Providers working together to commit fraud.
- **Prescription Fraud**: Falsifying prescriptions.
- **DRG Manipulation**: Misclassifying diagnoses to increase payments.
- **Modifier Abuse**: Incorrectly using modifiers to increase reimbursement.
- **Place-of-Service Abuse**: Billing for services in a more expensive setting than provided.
- **Excessive Billing**: Charging more than is reasonable for services.
- **Overutilization**: Providing more services than necessary.
- **Underutilization**: Failing to provide necessary services.
- **Abnormal Provider Behavior**: Unusual patterns in billing or service provision.
- **Suspicious Patient Behavior**: Patients exhibiting unusual patterns in claims.
- **Abnormal Billing Patterns**: Claims that deviate from normal billing practices.
- **Outlier Claims**: Claims that are significantly higher than average.
- **Impossible Day Billing**: Billing for services on days when they cannot occur.
- **Unusual Diagnosis-Procedure Combinations**: Combinations that do not typically occur together.
- **Geographic Anomalies**: Billing patterns that are unusual for a specific location.
- **Provider Specialty Mismatch**: Services billed by providers not qualified to provide them.

## 6. How to Detect Fraud or Anomalies
Detection methods include:
- **Rule-Based Detection**: Using predefined rules to identify fraud.
- **Statistical Analysis**: Analyzing data for unusual patterns.
- **Peer-Group Comparison**: Comparing providers to their peers.
- **Outlier Detection**: Identifying claims that deviate significantly from the norm.
- **Isolation Forest**: An algorithm for anomaly detection.
- **Local Outlier Factor**: Identifying anomalies based on local density.
- **Logistic Regression**: Predicting the likelihood of fraud.
- **Random Forest**: A machine learning method for classification.
- **XGBoost**: An optimized gradient boosting library.
- **Neural Networks**: Deep learning models for complex pattern recognition.
- **Time-Series Analysis**: Analyzing data over time for trends.
- **Graph Analytics**: Analyzing relationships between entities.
- **Provider Network Analysis**: Examining connections between providers.
- **Natural Language Processing (NLP)**: Analyzing text data for insights.
- **Large Language Models (LLM)**: Advanced NLP techniques for understanding claims.
- **Retrieval-Augmented Generation (RAG)**: Combining retrieval and generation for insights.
- **Knowledge Graphs**: Visualizing relationships between data points.
- **Explainable AI**: Ensuring transparency in AI decision-making.
- **Human-in-the-Loop Review**: Combining AI with human oversight for accuracy.

## 7. Useful Machine Learning Features
Key features for machine learning models include:
- **Provider ID / NPI**: Unique identifiers for healthcare providers.
- **Patient ID**: Unique identifiers for patients.
- **Diagnosis Codes**: ICD codes for diagnoses.
- **Procedure Codes**: CPT/HCPCS codes for services.
- **DRG**: Diagnosis-Related Group codes.
- **Revenue Code**: Codes for billing purposes.
- **NDC**: National Drug Codes for medications.
- **Claim Amount**: Total amount billed.
- **Paid Amount**: Amount paid by insurance.
- **Service Date**: Date services were provided.
- **Provider Specialty**: Type of services provided by the provider.
- **Geography**: Location of service provision.
- **Billing Frequency**: How often claims are submitted.
- **Historical Utilization**: Past usage patterns.
- **Referral Patterns**: Patterns in patient referrals.
- **Denial History**: History of claim denials.
- **Medical Necessity Indicators**: Indicators of whether services were necessary.

## 8. Useful CMS and Healthcare Data Sources
Key data sources include:
- **CMS Claims Data**: Data on claims submitted to CMS.
- **Medicare Data**: Data specific to Medicare beneficiaries.
- **Medicaid Data**: Data specific to Medicaid beneficiaries.
- **NPI Registry**: Database of healthcare providers.
- **PECOS**: Provider Enrollment, Chain, and Ownership System.
- **ICD-10**: International Classification of Diseases codes.
- **CPT**: Current Procedural Terminology codes.
- **HCPCS**: Healthcare Common Procedure Coding System codes.
- **DRG**: Diagnosis-Related Group codes.
- **Revenue Codes**: Codes for billing purposes.
- **NDC**: National Drug Codes for medications.
- **Provider Enrollment Data**: Data on provider participation.
- **OIG Exclusion List**: List of excluded providers.
- **Beneficiary Data**: Data on beneficiaries enrolled in CMS programs.
- **Payment Policy Documents**: Guidelines for payment policies.

## 9. AI and Business Applications
AI and machine learning support various applications, including:
- **Fraud Detection**: Identifying fraudulent claims.
- **Claims Anomaly Detection**: Detecting unusual claims patterns.
- **Payment Integrity**: Ensuring accurate payments.
- **Claims Auditing**: Reviewing claims for compliance.
- **Provider Risk Scoring**: Assessing provider risk levels.
- **Compliance Monitoring**: Ensuring adherence to regulations.
- **Investigation Support**: Assisting in fraud investigations.
- **Healthcare Analytics**: Analyzing healthcare data for insights.
- **RAG Chatbot**: Providing automated responses for inquiries.
- **LLM Healthcare Assistant**: Assisting with healthcare-related queries.
- **Human Investigator Workflow**: Streamlining the investigation process.

---

For more information, visit the [CMS website](https://www.cms.gov).
```
This brochure provides a comprehensive overview of CMS Healthcare Claims Fraud & Anomaly Detection, designed to be accessible to a wide audience, including healthcare data scientists, fraud analysts, compliance teams, AI engineers, and non-technical healthcare stakeholders.

## Step 11: Save the brochure

In [21]:
def save_brochure(brochure, filename="CMS_Healthcare_Fraud_Anomaly_Detection_Brochure.md"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(brochure)

    print("Brochure saved to:", filename)

In [22]:
save_brochure(brochure)

Brochure saved to: CMS_Healthcare_Fraud_Anomaly_Detection_Brochure.md


## Step 12: Next steps

This brochure generator can be extended into a full healthcare AI project:

- Add more CMS pages
- Store content in a vector database
- Build a RAG chatbot
- Add claims data
- Engineer fraud/anomaly features
- Train anomaly detection models
- Build a dashboard
- Add LLM-based investigator summaries